In [2]:
import pandas as pd
import altair as alt

# 1. Leer la base de datos
df = pd.read_csv('BASE DE DATOS(Hoja1).csv', encoding='latin1')

# 2. Encontrar el año de debut de hombres y mujeres por deporte
# Filtramos donde haya al menos 1 prueba masculina y sacamos el año mínimo
df_hombres = df[df['N°PRUEBAS HOMBRES'] > 0].groupby('DEPORTE')['AÑO'].min().reset_index()
df_hombres = df_hombres.rename(columns={'AÑO': 'Debut_Masculino'})

# Filtramos donde haya al menos 1 prueba femenina y sacamos el año mínimo
df_mujeres = df[df['N° PRUEBAS MUJERES'] > 0].groupby('DEPORTE')['AÑO'].min().reset_index()
df_mujeres = df_mujeres.rename(columns={'AÑO': 'Debut_Femenino'})

# 3. Unir ambos datos y calcular la "Brecha de Espera" (Años de retraso)
df_brecha = pd.merge(df_hombres, df_mujeres, on='DEPORTE', how='inner')
df_brecha['Años_de_Espera'] = df_brecha['Debut_Femenino'] - df_brecha['Debut_Masculino']

# 4. Filtrar y ordenar: Nos quedamos con deportes que tuvieron retraso y sacamos el Top 15 más desigual
df_brecha = df_brecha[df_brecha['Años_de_Espera'] > 0]
df_top15 = df_brecha.sort_values('Años_de_Espera', ascending=False).head(15)

# 5. Crear el gráfico de barras horizontales con Altair
grafico = alt.Chart(df_top15).mark_bar(color='#E36B73', cornerRadiusEnd=4).encode(
    x=alt.X('Años_de_Espera:Q', title='Años de espera para el debut femenino'),
    y=alt.Y('DEPORTE:N', sort='-x', title='Deporte Olímpico'), # sort='-x' ordena de mayor a menor
    tooltip=[
        alt.Tooltip('DEPORTE', title='Deporte'),
        alt.Tooltip('Debut_Masculino', title='Año debut Hombres'),
        alt.Tooltip('Debut_Femenino', title='Año debut Mujeres'),
        alt.Tooltip('Años_de_Espera', title='Años de retraso')
    ]
).properties(
    title='La sala de espera: Los deportes olímpicos que más años marginaron a las mujeres',
    width=700,
    height=450
)

# 6. Mostrar y exportar
grafico.display()
grafico.save('vis_03.html')

alt.Chart(...)